In [1]:
#Validating daily bias corrected rasters against daily raw products using test stations

#Libraries
import pandas as pd
import numpy as np
import glob
import os
import rasterio 
import HydroErr as he
import concurrent.futures
from pyproj import Transformer

# Paths
insitu_csv    = r"..\Data\Processed\Insitu\insitu_rainfall_1991_2020.csv"
train_stations = r"..\Data\Processed\Split_stations\train_stations.csv"
test_stations  = r"..\Data\Processed\Split_stations\test_stations.csv"
chirps_path    = r"..\Data\Datasets\CHIRPS_prec"
persiann_path  = r"..\Data\Datasets\PERSIANN_prec"
mswep_path     = r"..\Data\Datasets\MSWEP_prec"
har_path       = r"..\Data\Datasets\HAR_prec"
era5l_path     = r"..\Data\Datasets\ERA5L_prec"

bc_path        = r"..\Data\Processed\Bias_correction_daily\Predicted_rasters\daily_predictions"
is_k_path = r"..\Data\Processed\Bias_correction_daily\Insitu_kriged_daily"
output_dir = r"..\Data\Processed\Bias_correction_daily\FinalValidation"
os.makedirs(output_dir, exist_ok= True)

## PARAMETERS
datasets   = ['CHIRPS', 'PERSIANN', 'HAR', 'MSWEP', 'ERA5L', 'BC', 'IS_K']

start_year, start_month = 2001, 1
end_year,   end_month   = 2005, 12
start_date = pd.Timestamp(f"{start_year}-{start_month:02d}-01")
end_date   = pd.Timestamp(f"{end_year}-{end_month:02d}-01") + pd.offsets.MonthEnd(0)

dataset_paths = {
    'CHIRPS':   chirps_path,
    'PERSIANN': persiann_path,
    'MSWEP':    mswep_path,
    'HAR':      har_path,
    'ERA5L':    era5l_path,
    'BC':       bc_path,
    'IS_K': is_k_path
}

In [2]:
# Read insitu data

insitu_raw = pd.read_csv(insitu_csv)
insitu_raw["date"]  = pd.to_datetime(insitu_raw[["year", "month", "day"]])
insitu_raw  = insitu_raw.rename(columns={"lat": "y", "long": "x",
    "rainfall_sum": "insitu_prec"})
insitu_raw  = insitu_raw[insitu_raw["insitu_prec"] >= 0] 

# Filter for study period
insitu_daily = insitu_raw[
    (insitu_raw["year"]  >= start_year) &
    (insitu_raw["year"]  <= end_year)
].copy()
insitu_daily[["x", "y"]] = insitu_daily[["x", "y"]].round(5)

# Train / test split
train_ids = set(pd.read_csv(train_stations)["index_no"])
test_ids  = set(pd.read_csv(test_stations)["index_no"])

insitu_train_daily = insitu_daily[insitu_daily["index_no"].isin(train_ids)].copy()
insitu_test_daily =  insitu_daily[insitu_daily["index_no"].isin(test_ids)].copy()

In [3]:
# build results by datasets

def extract_date(filepath, dataset):
    """Parse date from filename for both daily and monthly naming conventions."""
    fname = os.path.basename(filepath)
    if dataset in ['BC']:
        # e.g. Calibrated_Raster_2001_01_01.tif → fname[-13:-4] = '2001_01_01'
        return pd.to_datetime(fname[-14:-4], format='%Y_%m_%d')
    else:
        # e.g. CHIRPS_20010115.tif → fname[-12:-4] = '20010115'
        return pd.to_datetime(fname[-12:-4], format='%Y%m%d')


def build_dataset_results(dataset_paths, datasets):
    """Build {dataset}_files dict pre-filtered to analysis period."""
    dataset_results = {}
    for dataset in datasets:
        files = sorted(glob.glob(os.path.join(dataset_paths[dataset], "*.tif")))
        filtered = []
        for f in files:
            try:
                d = extract_date(f, dataset)
                if start_date <= d <= end_date:
                    filtered.append(f)
            except Exception as e:
                print(f"  Skipping {os.path.basename(f)}: {e}")
        dataset_results[f'{dataset}_files'] = filtered
        print(f"{dataset}: {len(filtered)} files in period")
    return dataset_results


# Unique test stations — order must stay fixed (iloc used in sampling)
unique_stn = (
    insitu_test_daily[['index_no', 'x', 'y']]
    .drop_duplicates('index_no')
    .reset_index(drop=True)
)
stn_points = list(zip(unique_stn['x'], unique_stn['y']))  # (lon, lat)

dataset_results = build_dataset_results(dataset_paths, datasets)

CHIRPS: 1826 files in period
PERSIANN: 1826 files in period
HAR: 1826 files in period
MSWEP: 1826 files in period
ERA5L: 1825 files in period
BC: 1796 files in period
IS_K: 1826 files in period


In [4]:
# %%
def _sample_file(file, dataset, stn_points, unique_stn):
    """Sample all station points from a single raster file."""
    date_obj = extract_date(file, dataset)
    try:
        with rasterio.open(file) as src:
            if src.crs.to_epsg() != 4326:
                transformer = Transformer.from_crs(
                    "EPSG:4326", src.crs, always_xy=True
                )
                pts = [transformer.transform(x, y) for x, y in stn_points]
            else:
                pts = stn_points

            sampled = list(src.sample(pts))

        return [
            {
                'date':              date_obj,
                'stn_id':            unique_stn.iloc[i]['index_no'],
                f'{dataset}_prec':   max(v[0], 0),
            }
            for i, v in enumerate(sampled)
        ]
    except Exception as e:
        print(f"  Error processing {os.path.basename(file)}: {e}")
        return []


def extract_and_aggregate_daily(dataset_results, stn_points, unique_stn, datasets):
    
    df_final = None

    for dataset in datasets:
        files = dataset_results.get(f'{dataset}_files', [])
        if not files:
            print(f"Warning: no files found for {dataset}, skipping.")
            continue
        print(f"Processing {dataset} ({len(files)} daily files)...")

        records = []
        with concurrent.futures.ThreadPoolExecutor() as executor:
            futures = executor.map(
                lambda f: _sample_file(f, dataset, stn_points, unique_stn), files
            )
            for result in futures:
                records.extend(result)

        df_dataset = pd.DataFrame(records)
        if df_dataset.empty:
            print(f"  Warning: no data extracted for {dataset}")
            continue

        df_final = (
            df_dataset if df_final is None
            else pd.merge(df_final, df_dataset,
                          on=['stn_id', 'date'], how='outer')
        )
        print(f"  → merged shape: {df_final.shape}")    
    return df_final

In [5]:
# get daily grid from the datasets 
df_grid_daily   = extract_and_aggregate_daily(
    dataset_results, stn_points, unique_stn, datasets
)

# Merge with insitu test observations
df_validation = pd.merge(
    insitu_test_daily.rename(columns={'index_no': 'stn_id'}),
    df_grid_daily,
    on=['stn_id', 'date'],
    how='inner'
)

print(f"Validation df shape: {df_validation.shape}")
print(f"Stations: {df_validation['stn_id'].nunique()}")
print(f"Columns: {df_validation.columns.tolist()}")
print(df_validation.head())

df_validation.to_csv(os.path.join(output_dir,r"validation_df.csv"), index= False)

Processing CHIRPS (1826 daily files)...
  → merged shape: (32868, 3)
Processing PERSIANN (1826 daily files)...
  → merged shape: (32868, 4)
Processing HAR (1826 daily files)...
  → merged shape: (32868, 5)
Processing MSWEP (1826 daily files)...
  → merged shape: (32868, 6)
Processing ERA5L (1825 daily files)...
  → merged shape: (32868, 7)
Processing BC (1796 daily files)...
  → merged shape: (32868, 8)
Processing IS_K (1826 daily files)...
  → merged shape: (32868, 9)
Validation df shape: (31569, 18)
Stations: 18
Columns: ['stn_id', 'station_name', 'district', 'y', 'x', 'elevation', 'year', 'month', 'day', 'insitu_prec', 'date', 'CHIRPS_prec', 'PERSIANN_prec', 'HAR_prec', 'MSWEP_prec', 'ERA5L_prec', 'BC_prec', 'IS_K_prec']
   stn_id       station_name district         y         x  elevation  year  \
0     102  Baitadi_Gothapani  Baitadi  29.55744  80.41317     1352.0  2001   
1     102  Baitadi_Gothapani  Baitadi  29.55744  80.41317     1352.0  2001   
2     102  Baitadi_Gothapani  Ba

In [6]:
#column normalization function
def normalize_column(col):
    return (col - col.min()) / (col.max() - col.min())
          
#calculate accuracy of datasets
def calculate_accuracy(dataframe_prec):
    df_accuracy = pd.DataFrame()

    for dataset in datasets:
        dataset_accuracy = {'dataset': dataset}

        #Convert to array
        prec_grid_array = dataframe_prec[f'{dataset}_prec'].values
        prec_insitu_array = dataframe_prec['insitu_prec'].values
    
        #drop null values, and negative values
        valid_indices = np.logical_and.reduce([
            ~np.isnan(prec_grid_array),
            ~np.isnan(prec_insitu_array),
            prec_grid_array >=  0,
            prec_insitu_array >= 0
        ])
    
        observations = np.sum(valid_indices)
        dataset_accuracy['observations_n'] = observations

        prec_grid_array = prec_grid_array[valid_indices]
        prec_insitu_array = prec_insitu_array[valid_indices]

        #correlation coefficient
        correlation = he.spearman_r(prec_grid_array, prec_insitu_array)
        dataset_accuracy['correlation'] = correlation

        #NRMSE
        nrmse = he.nrmse_mean(prec_grid_array, prec_insitu_array)
        dataset_accuracy['nrmse'] = nrmse

        #Percent bias
        pbias = 100 * np.sum(prec_grid_array - prec_insitu_array) / np.sum(prec_insitu_array)
        dataset_accuracy['pbias'] = pbias

        #Relative standard deviation 
        rsd = (np.std(prec_grid_array))/(np.std(prec_insitu_array))
        dataset_accuracy['rsd'] = rsd

        df_dataset_accuracy = pd.DataFrame([dataset_accuracy])
        df_accuracy = pd.concat([df_accuracy, df_dataset_accuracy], ignore_index=True)

    #Normalization
    df_accuracy['abs_pbias'] = df_accuracy['pbias'].abs()
    df_accuracy['diff_rsd'] = (df_accuracy['rsd']-1).abs()

    df_accuracy_normalized = df_accuracy.copy().drop(columns=['pbias'])

    metrics = ['correlation', 'nrmse', 'abs_pbias', 'diff_rsd']
    for column in metrics:
        df_accuracy_normalized[column] = normalize_column(df_accuracy_normalized[column])

    #converting to scale of 0 to 1 (ideal to worst) - correlation & rsd needs inversion
    df_accuracy_normalized['correlation'] = 1 - df_accuracy_normalized['correlation']

    #get cpi for each (linear)
    df_accuracy_normalized['cpi'] = df_accuracy_normalized['correlation'] + df_accuracy_normalized['nrmse'] + df_accuracy_normalized['abs_pbias'] + df_accuracy_normalized['diff_rsd']
    df_accuracy_normalized['cpi_rank'] = df_accuracy_normalized['cpi'].rank(ascending=True)

    df_accuracy_final = df_accuracy_normalized[['dataset','cpi','cpi_rank']].merge(df_accuracy[['dataset','correlation','nrmse','pbias','rsd', 'observations_n']], on=['dataset'], how='inner')
    return df_accuracy_final

#get daily data accuracy
df_accuracy = calculate_accuracy(df_validation)
print(df_accuracy)

    dataset       cpi  cpi_rank  correlation     nrmse      pbias       rsd  \
0    CHIRPS  2.373877       6.0     0.300577  3.192224 -19.561426  0.802848   
1  PERSIANN  1.937573       5.0     0.420752  2.863950 -30.075244  0.530971   
2       HAR  1.309336       4.0     0.518792  3.068829  -9.755353  0.814546   
3     MSWEP  3.367353       7.0     0.416669  3.034942 -90.080683  0.132381   
4     ERA5L  1.214160       3.0     0.554072  3.100003  21.360378  0.931655   
5        BC  0.720190       2.0     0.586860  2.499751 -21.004226  0.538923   
6      IS_K  0.461730       1.0     0.583282  2.530716  -0.507213  0.608335   

   observations_n  
0           31569  
1           31569  
2           31569  
3           31569  
4           31552  
5           31059  
6           31569  


In [7]:
# determine the same but for training stations data

# Unique TRAIN stations (mirrors the test station setup)
unique_stn_train = (
    insitu_train_daily[['index_no', 'x', 'y']]
    .drop_duplicates('index_no')
    .reset_index(drop=True)
)
stn_points_train = list(zip(unique_stn_train['x'], unique_stn_train['y']))  # (lon, lat)

# Extract grid values at training station locations
df_grid_daily_train = extract_and_aggregate_daily(
    dataset_results, stn_points_train, unique_stn_train, datasets
)

# Merge with insitu train observations
df_validation_train = pd.merge(
    insitu_train_daily.rename(columns={'index_no': 'stn_id'}),
    df_grid_daily_train,
    on=['stn_id', 'date'],
    how='inner'
)

# # Filter to months with sufficient insitu coverage (consistent with test set)
# df_validation_train = df_validation_train[df_validation_train['valid_days'] >= 25].copy()

# # print(f"Train validation df shape: {df_validation_train.shape}")
# print(f"Stations: {df_validation_train['stn_id'].nunique()}")
# print(f"Columns: {df_validation_train.columns.tolist()}")
# print(df_validation_train.head())

df_validation_train.to_csv(os.path.join(output_dir, "validation_train_df.csv"), index=False)

# Calculate accuracy for training stations
df_accuracy_train = calculate_accuracy(df_validation_train)
df_accuracy_train

Processing CHIRPS (1826 daily files)...
  → merged shape: (91300, 3)
Processing PERSIANN (1826 daily files)...
  → merged shape: (91300, 4)
Processing HAR (1826 daily files)...
  → merged shape: (91300, 5)
Processing MSWEP (1826 daily files)...
  → merged shape: (91300, 6)
Processing ERA5L (1825 daily files)...
  → merged shape: (91300, 7)
Processing BC (1796 daily files)...
  → merged shape: (91300, 8)
Processing IS_K (1826 daily files)...
  → merged shape: (91300, 9)


,dataset,cpi,cpi_rank,correlation,nrmse,pbias,rsd,observations_n
0,CHIRPS,2.036643,5.0,0.277560,3.286818,-8.394857,0.920439,88093
1,PERSIANN,2.149729,6.0,0.409958,2.836889,-24.330429,0.575822,88093
2,HAR,1.639394,4.0,0.509915,3.343364,19.953574,1.026979,88093
3,MSWEP,3.528909,7.0,0.406383,2.932853,-89.333553,0.139553,88093
4,ERA5L,1.621218,3.0,0.563271,3.180417,33.617484,1.030990,88047
5,BC,0.322060,1.0,0.739118,1.205133,-14.413934,0.766576,86683
6,IS_K,0.497505,2.0,0.639667,1.350590,17.517377,0.888590,88093


In [8]:
# Calculate categorical metrics

import xarray as xr
import operator
import scores

def calculate_categorical_metrics(dataframe_prec):
    df_categorical_metrics = pd.DataFrame()

    for dataset in datasets:
        dataset_categorical_metrics = {'dataset': dataset}

        # Convert to xarray (Note: using 'insitu_prec' to match df_validation)
        obs_insitu = dataframe_prec.set_index(['stn_id', 'date'])['insitu_prec'].to_xarray()
        pred_dataset = dataframe_prec.set_index(['stn_id', 'date'])[f'{dataset}_prec'].to_xarray()

        # Define event - wet day is when precipitation is >= 0.1
        event_operator = scores.categorical.ThresholdEventOperator(
            default_event_threshold=0.1, 
            default_op_fn=operator.ge
        )

        # Conversion to binary
        dataset_binary, insitu_binary = event_operator.make_event_tables(pred_dataset, obs_insitu)
            
        # Make contingency manager
        cm_dataset = event_operator.make_contingency_manager(dataset_binary, insitu_binary)
        
        # Calculate metrics
        pod_dataset = cm_dataset.probability_of_detection()
        dataset_categorical_metrics['pod'] = pod_dataset.values
        
        far_dataset = cm_dataset.false_alarm_rate()
        dataset_categorical_metrics['far'] = far_dataset.values
        
        fb_dataset = cm_dataset.frequency_bias() 
        dataset_categorical_metrics['fb'] = fb_dataset.values
        
        csi_dataset = cm_dataset.critical_success_index()
        dataset_categorical_metrics['csi'] = csi_dataset.values
        
        # Wrap in a DataFrame and append
        df_temp = pd.DataFrame([dataset_categorical_metrics])
        df_categorical_metrics = pd.concat([df_categorical_metrics, df_temp], ignore_index=True)

    return df_categorical_metrics

# Get daily categorical metrics from validation dataframe
df_categorical_metrics = calculate_categorical_metrics(df_validation)

# Format numeric columns
numeric_cols = ['pod', 'far', 'fb', 'csi']
df_categorical_metrics[numeric_cols] = df_categorical_metrics[numeric_cols].astype(float).round(2)

print("\nCategorical Metrics:")
print(df_categorical_metrics)

# Optional: Merge continuous and categorical metrics into a single summary table
df_overall_metrics = pd.merge(df_accuracy, df_categorical_metrics, on='dataset', how='inner')
df_overall_metrics.to_csv(os.path.join(output_dir, r"overall_metrics.csv"), index=False)


Categorical Metrics:
    dataset   pod   far    fb   csi
0    CHIRPS  0.36  0.12  0.68  0.27
1  PERSIANN  0.78  0.38  1.78  0.39
2       HAR  0.82  0.31  1.63  0.45
3     MSWEP  0.56  0.18  1.04  0.38
4     ERA5L  0.97  0.60  2.56  0.37
5        BC  0.67  0.12  1.00  0.51
6      IS_K  0.96  0.40  2.02  0.46
